# 02. Data Cleaning Techniques

**Aim:** Handle missing values, duplicates, corrupt images, and data type issues in the dataset manifest.

## Theory

Data cleaning helps improve dataset quality before analysis or modeling. Typical checks include missing values, duplicate rows, invalid file paths, and corrupt images that cannot be opened. Class labels may also contain formatting inconsistencies such as leading or trailing spaces or inconsistent casing. Detecting rare classes is also important because severe imbalance can affect downstream model performance and evaluation.

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('../logs/.mplconfig'))
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

TRAIN_DIR = Path('../data/PlantDoc-Dataset-master/train/')
TEST_DIR = Path('../data/PlantDoc-Dataset-master/test/')
MANIFEST_PATH = Path('../data/dataset_manifest.csv')
CLEAN_MANIFEST_PATH = Path('../data/dataset_manifest_clean.csv')
FEATURES_PATH = Path('../data/image_features.csv')

def scan_split(split_dir: Path, split_name: str) -> pd.DataFrame:
    records = []
    if not split_dir.exists():
        return pd.DataFrame(columns=['image_path', 'class_name', 'split'])
    for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        for image_path in sorted(class_dir.rglob('*')):
            if image_path.is_file():
                records.append({
                    'image_path': str(image_path.as_posix()),
                    'class_name': class_dir.name,
                    'split': split_name,
                })
    return pd.DataFrame(records)


def ensure_manifest(prefer_clean: bool = True) -> pd.DataFrame:
    target = CLEAN_MANIFEST_PATH if prefer_clean and CLEAN_MANIFEST_PATH.exists() else MANIFEST_PATH
    if target.exists():
        return pd.read_csv(target)

    train_df = scan_split(TRAIN_DIR, 'train')
    test_df = scan_split(TEST_DIR, 'test')
    full_df = pd.concat([train_df, test_df], ignore_index=True)
    full_df['class_name'] = full_df['class_name'].astype(str).str.strip()
    full_df = full_df.drop_duplicates(subset=['image_path']).reset_index(drop=True)
    return full_df


def validate_image(image_path: str):
    path = Path(image_path)
    if not path.exists():
        return False, 'missing'
    try:
        with Image.open(path) as img:
            img.verify()
        return True, 'ok'
    except Exception:
        return False, 'corrupt'


def load_rgb_image(image_path: str, size=(224, 224)):
    with Image.open(image_path) as img:
        rgb = img.convert('RGB').resize(size)
        return np.array(rgb)


def extract_rgb_features(image_array: np.ndarray) -> dict:
    channels = image_array.reshape(-1, 3)
    return {
        'mean_r': float(channels[:, 0].mean()),
        'mean_g': float(channels[:, 1].mean()),
        'mean_b': float(channels[:, 2].mean()),
        'std_r': float(channels[:, 0].std()),
        'std_g': float(channels[:, 1].std()),
        'std_b': float(channels[:, 2].std()),
    }


def ensure_features() -> pd.DataFrame:
    if FEATURES_PATH.exists():
        return pd.read_csv(FEATURES_PATH)

    manifest_df = ensure_manifest(prefer_clean=True).copy()
    quality_records = []
    for image_path in manifest_df['image_path']:
        valid, status = validate_image(image_path)
        quality_records.append((valid, status))
    manifest_df[['is_valid_image', 'file_status']] = pd.DataFrame(quality_records, index=manifest_df.index)
    manifest_df = manifest_df[manifest_df['is_valid_image']].copy().reset_index(drop=True)

    feature_rows = []
    for row in manifest_df.itertuples(index=False):
        image_array = load_rgb_image(row.image_path)
        feature_rows.append({
            'image_path': row.image_path,
            'class_name': row.class_name,
            'split': row.split,
            **extract_rgb_features(image_array),
        })

    features_df = pd.DataFrame(feature_rows)
    return features_df


## Code

In [2]:
manifest_df = ensure_manifest(prefer_clean=False).copy()
rows_before = len(manifest_df)

print('Initial shape:', manifest_df.shape)
print('\nNull values per column:')
print(manifest_df.isnull().sum())

manifest_df['is_duplicate_path'] = manifest_df.duplicated(subset=['image_path'])
print('\nDuplicate image paths:', int(manifest_df['is_duplicate_path'].sum()))

manifest_df['file_exists'] = manifest_df['image_path'].apply(lambda p: Path(p).exists())
quality_checks = manifest_df['image_path'].apply(validate_image)
manifest_df['is_valid_image'] = quality_checks.apply(lambda x: x[0])
manifest_df['file_status'] = quality_checks.apply(lambda x: x[1])

manifest_df['class_name_original'] = manifest_df['class_name'].astype(str)
manifest_df['class_name'] = manifest_df['class_name'].astype(str).str.strip()
manifest_df['class_name'] = manifest_df['class_name'].str.replace(r'\s+', ' ', regex=True)
manifest_df['class_name_standardized'] = manifest_df['class_name'].str.lower()

class_counts = manifest_df['class_name'].value_counts()
manifest_df['is_rare_spider_mite_class'] = manifest_df['class_name'].eq('Tomato two spotted spider mites leaf')

clean_df = manifest_df.loc[(~manifest_df['is_duplicate_path']) & manifest_df['file_exists'] & manifest_df['is_valid_image']].copy()
clean_df = clean_df.drop(columns=['is_duplicate_path'])
clean_df = clean_df.reset_index(drop=True)

clean_output = Path('../data/dataset_manifest_clean.csv')
clean_output.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(clean_output, index=False)

print('\nRare spider mite class count:', int(class_counts.get('Tomato two spotted spider mites leaf', 0)))
print('Saved cleaned manifest to:', clean_output)

Initial shape: (2572, 3)

Null values per column:
image_path    0
class_name    0
split         0
dtype: int64

Duplicate image paths: 0

Rare spider mite class count: 2
Saved cleaned manifest to: ../data/dataset_manifest_clean.csv


## Results & Evaluation

In [3]:
rows_after = len(clean_df)
missing_files = manifest_df.loc[~manifest_df['file_exists'], 'image_path'].tolist()
corrupt_files = manifest_df.loc[manifest_df['file_status'] == 'corrupt', 'image_path'].tolist()

print('Rows before cleaning:', rows_before)
print('Rows after cleaning:', rows_after)
print('Removed rows:', rows_before - rows_after)
print('\nMissing files found:')
print(missing_files if missing_files else 'None')
print('\nCorrupt files found:')
print(corrupt_files if corrupt_files else 'None')
print('\nCleaned manifest preview:')
display(clean_df.head())

Rows before cleaning: 2572
Rows after cleaning: 2572
Removed rows: 0

Missing files found:
None

Corrupt files found:
None

Cleaned manifest preview:


,image_path,class_name,split,file_exists,is_valid_image,file_status,class_name_original,class_name_standardized,is_rare_spider_mite_class
0,../data/PlantDoc-Dataset-master/train/Apple Sc...,Apple Scab Leaf,train,True,True,ok,Apple Scab Leaf,apple scab leaf,False
1,../data/PlantDoc-Dataset-master/train/Apple Sc...,Apple Scab Leaf,train,True,True,ok,Apple Scab Leaf,apple scab leaf,False
2,../data/PlantDoc-Dataset-master/train/Apple Sc...,Apple Scab Leaf,train,True,True,ok,Apple Scab Leaf,apple scab leaf,False
3,../data/PlantDoc-Dataset-master/train/Apple Sc...,Apple Scab Leaf,train,True,True,ok,Apple Scab Leaf,apple scab leaf,False
4,../data/PlantDoc-Dataset-master/train/Apple Sc...,Apple Scab Leaf,train,True,True,ok,Apple Scab Leaf,apple scab leaf,False


## Conclusion

The dataset manifest was checked for nulls, duplicates, missing files, and corrupt images, then standardized for consistent class naming. The cleaned manifest provides a safer foundation for feature extraction and modeling while clearly highlighting the extremely rare spider-mite class.